# CinLM — Colab G4 (RTX PRO 6000 Blackwell) Lab & Training Notebook

**Arquitetura:** Hybrid Attention 3:1 (SWA + Global, GQA, RoPE, QK-Norm) + Engram Conditional Memory + Sparse `ReLU²` MoE (1 shared + 8 routed top-2, aux-loss-free balancing), projetada para **competência em ~100M ativos (PT-BR)** e **interpretabilidade nativa por Frontier LLMs**.

### Fluxo deste notebook
1. Verificar GPU (G4 / A100) e instalar `cinlm`.
2. Rodar a suíte de testes científicos (causalidade, paridade KV+Engram streaming/full, MoE esparso, exatidão de Direct Logit Attribution).
3. Inspecionar a tabela exata de parâmetros da escada iso-ativa (~100M ativos).
4. Treinar o tokenizador BPE PT-BR 32k em streaming (`Polygl0t/gigaverbo-v2`).
5. Benchmark de throughput (tokens/s) na G4 e início do pré-treino com checkpoint automático no Google Drive.
6. Exportar JSON de auditoria (`run_frontier_audit`) + teste de intervenção causal (`InterventionSpec`).

In [ ]:
# 1. Verificar GPU (atenção: troque para G4 ou A100 em Runtime -> Change runtime type se aparecer T4!)
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
VRAM_GB = 0.0
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    VRAM_GB = props.total_memory / 1e9
    cc = torch.cuda.get_device_capability(0)
    print(f'GPU: {props.name} | VRAM: {VRAM_GB:.1f} GB | sm_{cc[0]}{cc[1]} | native BF16: {cc[0] >= 8}')
    if VRAM_GB < 20:
        print('⚠️ Você está numa GPU de ~15 GB (ex: T4). Funciona para testes com --grad-checkpoint e batch pequeno, mas para treino sério troque o runtime para G4 (96 GB) ou A100!')

In [ ]:
# 2. Sincronizar código via GitHub (Jshebb/CinLM) e montar Drive para checkpoints persistentes
import os
from pathlib import Path

REPO_URL = 'https://github.com/Jshebb/CinLM.git'
REPO_DIR = Path('/content/CinLM')

# Montar Google Drive para salvar checkpoints e tokenizer de forma persistente
USE_DRIVE = True
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        PERSIST_DIR = Path('/content/drive/MyDrive/CinLM')
    except Exception:
        PERSIST_DIR = Path('/content/CinLM_persist')
else:
    PERSIST_DIR = Path('/content/CinLM_persist')
PERSIST_DIR.mkdir(parents=True, exist_ok=True)
print('Diretório persistente:', PERSIST_DIR)

if (REPO_DIR / '.git').exists():
    !git -C /content/CinLM pull --ff-only
else:
    !git clone {REPO_URL} /content/CinLM

os.chdir(REPO_DIR)
import importlib, sys
src_path = str(REPO_DIR / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print('CWD atual:', os.getcwd())
!pip -q install tokenizers datasets pytest
!pip -q install --no-build-isolation --no-deps -e .
importlib.invalidate_caches()
!pytest -v

In [ ]:
# 3. Conferir tabela exata da escada iso-ativa (~100M ativos/token)
from cinlm.config import get_preset_config

for name in ['dense_global_100m', 'dense_hybrid_100m', 'hybrid_engram_100m', 'full_260m_a100m']:
    cfg = get_preset_config(name)
    est = cfg.estimate_parameters()
    print(f"{name:20s} | total={est['total']/1e6:6.2f}M | active={est['active_per_token']/1e6:6.2f}M | engram_tables={est['engram_tables']/1e6:5.2f}M")

In [ ]:
# 4. Treinar tokenizador BPE PT-BR 32k em streaming (~3-5 min em CPU) e salvar no Drive
TOK_PATH = PERSIST_DIR / 'cinlm_bpe_32k.json'
if not TOK_PATH.exists():
    !python scripts/train_tokenizer.py --dataset Polygl0t/gigaverbo-v2 --max-docs 150000 --vocab-size 32768 --output {TOK_PATH}
else:
    print('Tokenizer já existe em:', TOK_PATH)

In [ ]:
# 5. Smoke / benchmark de throughput (ajusta batch automaticamente para A100-80GB/G4, A100-40GB, L4, T4)
if VRAM_GB >= 70:
    BS, GA, EXTRA = 32, 2, ''
elif VRAM_GB >= 35:
    BS, GA, EXTRA = 16, 4, ''
elif VRAM_GB >= 20:
    BS, GA, EXTRA = 4, 16, ''
else:
    BS, GA, EXTRA = 2, 8, '--grad-checkpoint'
print(f'Benchmark config (VRAM={VRAM_GB:.1f}GB): batch_size={BS}, grad_accum={GA}, extra={EXTRA}')
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/train.py --preset full_260m_a100m --synthetic --seq-len 2048 --batch-size {BS} --grad-accum {GA} --max-steps 10 --log-every 5 {EXTRA} --out-dir /content/g4_bench

In [ ]:
# 6. Pré-treino real em PT-BR (streaming GigaVerbo-v2 + Synth + FineWeb-Edu)
# Salva checkpoints direto no Drive (PERSIST_DIR) com --auto-resume para nunca perder progresso!
RUN_DIR = PERSIST_DIR / 'runs' / 'cinlm_full_ptbr'
if VRAM_GB >= 70:
    BS, GA, EXTRA = 32, 2, ''
elif VRAM_GB >= 35:
    BS, GA, EXTRA = 16, 4, ''
elif VRAM_GB >= 20:
    BS, GA, EXTRA = 4, 16, ''
else:
    BS, GA, EXTRA = 2, 16, '--grad-checkpoint'
print(f'Train config (VRAM={VRAM_GB:.1f}GB): batch_size={BS}, grad_accum={GA}, tokens/step={BS*GA*2048:,}')
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/train.py \
  --preset full_260m_a100m \
  --tokenizer {TOK_PATH} \
  --seq-len 2048 --batch-size {BS} --grad-accum {GA} \
  --max-steps 10000 --warmup-steps 250 --save-every 250 --keep-checkpoints 2 --auto-resume \
  {EXTRA} --out-dir {RUN_DIR}

In [ ]:
# 7. Auditoria de Interpretabilidade para Frontier LLM (antes e depois de ablação do Engram)
import torch
from tokenizers import Tokenizer
from cinlm import CinLM, get_preset_config, InterventionSpec, run_frontier_audit

tok = Tokenizer.from_file(str(TOK_PATH))
cfg = get_preset_config('full_260m_a100m')
model = CinLM(cfg).cuda().eval()
# Para usar um checkpoint treinado:
# ckpt = torch.load('runs/cinlm_full_ptbr/ckpt_step_5000.pt', map_location='cuda', weights_only=False)
# model.load_state_dict(ckpt['model'])

prompt = 'A capital do Brasil é'
ids = torch.tensor([tok.encode(prompt).ids], dtype=torch.long, device='cuda')

report_clean = run_frontier_audit(model, ids, decode_token=lambda i: tok.decode([i]))
report_no_engram = run_frontier_audit(
    model, ids, decode_token=lambda i: tok.decode([i]),
    intervention=InterventionSpec(force_engram_gate={l: 0.0 for l in cfg.engram_layers}),
)
print('Top-3 normal:', report_clean.top_predictions[:3])
print('Top-3 sem Engram:', report_no_engram.top_predictions[:3])
print('\nTop-5 Direct Logit Attribution:')
for item in report_clean.direct_logit_attribution[:5]:
    print(' ', item)